In [ ]:
import pandas as pd
from pathlib import Path

# ── Configure your CSV paths here ──────────────────────────────
CSV_PATHS = [
    "submission_cached_finetune_b.csv",
    "submission_cached_finetune.csv",
    "submission.csv",
]

IMAGE_COL  = "image_id"
CLUSTER_COL = "cluster"
#

dfs = [pd.read_csv(p) for p in CSV_PATHS]
for i, df in enumerate(dfs):
    print(f"CSV {i+1}: {len(df)} rows | clusters: {df[CLUSTER_COL].nunique()}")

CSV 1: 2409 rows | clusters: 1110
CSV 2: 2409 rows | clusters: 1267
CSV 3: 2409 rows | clusters: 1130


In [2]:
from scipy.stats import mode

# Merge all CSVs on image_id
merged = dfs[0][[IMAGE_COL, CLUSTER_COL]].rename(columns={CLUSTER_COL: "cluster_0"})
for i, df in enumerate(dfs[1:], 1):
    merged = merged.merge(
        df[[IMAGE_COL, CLUSTER_COL]].rename(columns={CLUSTER_COL: f"cluster_{i}"}),
        on=IMAGE_COL, how="outer"
    )

cluster_cols = [c for c in merged.columns if c.startswith("cluster_")]

def majority_vote(row):
    votes = row[cluster_cols].dropna()
    if len(votes) == 0:
        return None
    return votes.value_counts().idxmax()

merged["majority_cluster"] = merged.apply(majority_vote, axis=1)

# Pass the winner explicitly — don't look it up from the row
def agreement(row):
    votes = row[cluster_cols].dropna()
    if len(votes) == 0:
        return None
    winner = row["majority_cluster"]  # ← now safe: majority_cluster already exists in merged
    return (votes == winner).mean()

merged["agreement"] = merged.apply(agreement, axis=1)

result = merged[[IMAGE_COL] + cluster_cols + ["majority_cluster", "agreement"]]
result.to_csv("majority_vote_result.csv", index=False)

print(f"✅ Saved majority_vote_result.csv ({len(result)} rows)")
print(f"Full agreement (100%): {(result['agreement'] == 1).sum()} images")
print(f"Partial agreement:     {(result['agreement'].between(0.01, 0.99)).sum()} images")
print(f"No consensus:         {result['agreement'].isna().sum()} images")
result.head(10)

✅ Saved majority_vote_result.csv (2409 rows)
Full agreement (100%): 575 images
Partial agreement:     1834 images
No consensus:         0 images


,image_id,cluster_0,cluster_1,cluster_2,majority_cluster,agreement
0,3,cluster_LynxID2025_11,cluster_LynxID2025_11,cluster_LynxID2025_11,cluster_LynxID2025_11,1.000000
1,5,cluster_LynxID2025_75,cluster_LynxID2025_new_68,cluster_LynxID2025_new_82,cluster_LynxID2025_75,0.333333
2,12,cluster_LynxID2025_new_0,cluster_LynxID2025_95,cluster_LynxID2025_95,cluster_LynxID2025_95,0.666667
3,13,cluster_LynxID2025_11,cluster_LynxID2025_11,cluster_LynxID2025_11,cluster_LynxID2025_11,1.000000
4,18,cluster_LynxID2025_11,cluster_LynxID2025_11,cluster_LynxID2025_11,cluster_LynxID2025_11,1.000000
5,19,cluster_LynxID2025_new_1,cluster_LynxID2025_new_0,cluster_LynxID2025_new_0,cluster_LynxID2025_new_0,0.666667
6,27,cluster_LynxID2025_25,cluster_LynxID2025_25,cluster_LynxID2025_25,cluster_LynxID2025_25,1.000000
7,33,cluster_LynxID2025_62,cluster_LynxID2025_95,cluster_LynxID2025_62,cluster_LynxID2025_62,0.666667
8,36,cluster_LynxID2025_54,cluster_LynxID2025_new_104,cluster_LynxID2025_32,cluster_LynxID2025_54,0.333333
9,45,cluster_LynxID2025_new_2,cluster_LynxID2025_new_125,cluster_LynxID2025_new_130,cluster_LynxID2025_new_2,0.333333


In [3]:
import pandas as pd

# Load your majority vote CSV
df = pd.read_csv("majority_vote_result.csv")

# Keep only needed columns and rename
final_df = df[["image_id", "majority_cluster"]].rename(
    columns={"majority_cluster": "cluster"}
)

# Save if needed
final_df.to_csv("final_submission.csv", index=False)

# Preview
print(final_df.head())

   image_id                cluster
0         3  cluster_LynxID2025_11
1         5  cluster_LynxID2025_75
2        12  cluster_LynxID2025_95
3        13  cluster_LynxID2025_11
4        18  cluster_LynxID2025_11
